In [ ]:
import os
os.environ['GROQ_API_KEY']  

#Install libraries

In [2]:
!pip install -U langchain-text-splitters


In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-groq faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
!pip install langchain_huggingface

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_757/3703732251.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


#Step 1a - Indexing (Document Ingestion)

In [12]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

try:
    video_id = "Gfr50f6ZBvo"  # Only the video ID, no &t

    ytt_api = YouTubeTranscriptApi()

    # Get first available transcript
    transcript = next(iter(ytt_api.list(video_id)))

    # Fetch transcript data
    transcript_data = transcript.fetch()

    # Convert to plain text
    clean_text = " ".join(snippet.text for snippet in transcript_data)

    print(clean_text)

except TranscriptsDisabled:
    print("No captions available for this video.")

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to 

#Step 1b - Indexing(Text splitting)

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks =splitter.create_documents([clean_text])

NameError: name 'clean_text' is not defined

In [ ]:
len(chunks)

In [ ]:
chunks[0]

#Step 1c & Indexing (Embedding Generation and Storing in vector store)

In [ ]:
embedding  = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(chunks,embedding)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['a9111ed2-c286-477b-b10c-c4064e111d0f'])

#Step 2 Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is deepmind')

#Step 3 Augmentation

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b" , temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template= """
    You are a helpful assistant
    Answer ONLY from the provided transcript context.
    if the context is insufficient, just say you dont know.

    {context}
    Question: {question}
    """,
    input_variables=["context","question"]

)

In [ ]:
question = "is the topic of aliens discussed in this video? if yes then what was discussed"
retriever_docs = retriever.invoke(question)

In [ ]:
retriever_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retriever_docs)

In [ ]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [ ]:
final_prompt

#Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

#Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')